<a href="https://colab.research.google.com/github/Aravindr017/Natural_Language_Processing-Spam_Classification/blob/main/Model/Spam_Classification_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing Libraries

In [394]:
!pip install gensim

In [395]:
import nltk
import pandas as pd
import numpy as np
import string # because we are handling messages and we have to convert to lower case

# for stemming
from nltk.stem import PorterStemmer
# for lemmatization
from nltk.stem import WordNetLemmatizer

# for bag of words
from sklearn.feature_extraction.text import CountVectorizer

# for train test split
from sklearn.model_selection import train_test_split

# for label encoding
from sklearn.preprocessing import LabelEncoder

# for Logistic regression model - classification
from sklearn.linear_model import LogisticRegression

# for evaluation metrices
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

# for ensemble (boosting model)
from sklearn.ensemble import AdaBoostClassifier

from sklearn.feature_extraction.text import TfidfVectorizer   # TF-IDF

# for N-gram
from sklearn.feature_extraction.text import CountVectorizer

# for word 2 vec
import gensim.downloader as api

# for CBOW
from gensim.models import Word2Vec

# Read Data

In [396]:
file_path = '/content/drive/MyDrive/ICT - Ai Ml/Natural Language Processing/Data/spam.xlsx'
df_spam = pd.read_excel(file_path)
df_spam.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# EDA

In [397]:
df_spam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [398]:
# relevent data is present in first two columns only
df_spam = df_spam[['v1', 'v2']]
df_spam.head(3)

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...


In [399]:
df_spam['v1'].value_counts()

,count
v1,
ham,4825
spam,747


# Preprocessing - on sample text

## Punctuations Removal using user defined function

In [400]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [401]:
sample_text = "Hello! + Can we =test #1 ($50.99) _item? Yes, @John_Doe said: 'It's a -well-known state-of-the-art 50% / 100% match [id: #789]—or is it?' Check example.com {file: data.csv, status: active | pending} <ok?> ~all done^;"
sample_text

"Hello! + Can we =test #1 ($50.99) _item? Yes, @John_Doe said: 'It's a -well-known state-of-the-art 50% / 100% match [id: #789]—or is it?' Check example.com {file: data.csv, status: active | pending} <ok?> ~all done^;"

In [402]:
# function to remove punctations from string

def remove_punctuation(text):
  punctuationless_text = ''.join([i for i in text if i not in string.punctuation])
  # comparing each character in 'text' against the punctuation list.
  return punctuationless_text

In [403]:
print('before')
print('--------------------')
print(sample_text)
print('\n\n')
print('After')
print('--------------------')
punctuation_free_text = remove_punctuation(sample_text)
print(remove_punctuation(sample_text))

before
--------------------
Hello! + Can we =test #1 ($50.99) _item? Yes, @John_Doe said: 'It's a -well-known state-of-the-art 50% / 100% match [id: #789]—or is it?' Check example.com {file: data.csv, status: active | pending} <ok?> ~all done^;



After
--------------------
Hello  Can we test 1 5099 item Yes JohnDoe said Its a wellknown stateoftheart 50  100 match id 789—or is it Check examplecom file datacsv status active  pending ok all done


## Lowercasing

In [404]:

print('before')
print('--------------------')
print(punctuation_free_text)
print('\n\n')
print('After')
print('--------------------')
lowercase_text = punctuation_free_text.lower()  # convert the string to lower case
print(lowercase_text)

before
--------------------
Hello  Can we test 1 5099 item Yes JohnDoe said Its a wellknown stateoftheart 50  100 match id 789—or is it Check examplecom file datacsv status active  pending ok all done



After
--------------------
hello  can we test 1 5099 item yes johndoe said its a wellknown stateoftheart 50  100 match id 789—or is it check examplecom file datacsv status active  pending ok all done


## Tokenization

In [405]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [406]:
# we are converting the entire text into a list of unique words
# function to get tokens from the sentance
# make sure the i/p text should be punctuation removed and in lower case.
def tokenization(text):
  words_list = nltk.word_tokenize(text)
  return words_list

In [407]:
tokens = tokenization(lowercase_text)
tokens

['hello',
 'can',
 'we',
 'test',
 '1',
 '5099',
 'item',
 'yes',
 'johndoe',
 'said',
 'its',
 'a',
 'wellknown',
 'stateoftheart',
 '50',
 '100',
 'match',
 'id',
 '789—or',
 'is',
 'it',
 'check',
 'examplecom',
 'file',
 'datacsv',
 'status',
 'active',
 'pending',
 'ok',
 'all',
 'done']

## Stop Words Removal

In [408]:
nltk.download('stopwords')    # library for stopwords
stop_words_list = nltk.corpus.stopwords.words('english')
stop_words_list

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [409]:
# function to remove the stop words in the sentence after tokentization
def remove_stopwords(tokens):
  stop_words = nltk.corpus.stopwords.words('english')
  tokens_without_stopwords = [i for i in tokens if i not in stop_words]
  return tokens_without_stopwords

In [410]:
clean_tokens = remove_stopwords(tokens)

print('before')
print('--------------------')
print(tokens)
print(f'There are {len(tokens)} tokens in this sentence')
print('\n\n')
print('After')
print('--------------------')
print(clean_tokens)
print(f'There are {len(clean_tokens)} tokens in this sentence')


before
--------------------
['hello', 'can', 'we', 'test', '1', '5099', 'item', 'yes', 'johndoe', 'said', 'its', 'a', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'is', 'it', 'check', 'examplecom', 'file', 'datacsv', 'status', 'active', 'pending', 'ok', 'all', 'done']
There are 31 tokens in this sentence



After
--------------------
['hello', 'test', '1', '5099', 'item', 'yes', 'johndoe', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'status', 'active', 'pending', 'ok', 'done']
There are 24 tokens in this sentence


## Stemming

- Removing the prefix and sufix and only consider the root word. <br>
eg : going -> go

In [411]:
stem_obj = PorterStemmer()

def stemming(clean_tokens):
  stem_list = [stem_obj.stem(word) for word in clean_tokens]
  return stem_list



# stem_obj.stem(word) - will give the stem word for each word in the clean_tokens

In [412]:
stem_token_list = stemming(clean_tokens)

print('before')
print('--------------------')
print(clean_tokens)
print(f'There are {len(clean_tokens)} tokens in this sentence')
print('\n\n')
print('After')
print('--------------------')
print(stem_token_list)
print(f'There are {len(stem_token_list)} tokens in this sentence')

before
--------------------
['hello', 'test', '1', '5099', 'item', 'yes', 'johndoe', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'status', 'active', 'pending', 'ok', 'done']
There are 24 tokens in this sentence



After
--------------------
['hello', 'test', '1', '5099', 'item', 'ye', 'johndo', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'statu', 'activ', 'pend', 'ok', 'done']
There are 24 tokens in this sentence


## Lemmatization

- Taking the root word with the meaning of the word <br>
eg : went -> go

In [413]:
nltk.download('wordnet')  # for lemmatization ( get words with meaning )

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [414]:
lemma_obj = WordNetLemmatizer()

def lemmatization(token):
  lemma_list = [lemma_obj.lemmatize(i) for i in token]
  return lemma_list

In [415]:
lemma_list = lemmatization(stem_token_list)

print('before')
print('--------------------')
print(stem_token_list)
print('\n\n')
print('After')
print('--------------------')
print(lemma_list)

before
--------------------
['hello', 'test', '1', '5099', 'item', 'ye', 'johndo', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'statu', 'activ', 'pend', 'ok', 'done']



After
--------------------
['hello', 'test', '1', '5099', 'item', 'ye', 'johndo', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'statu', 'activ', 'pend', 'ok', 'done']


# Preprocessing - on Dataset

##  Convert to lower case  

In [416]:
df_spam['v2'].head(10)

,v2
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."
5,FreeMsg Hey there darling it's been 3 week's n...
6,Even my brother is not like to speak with me. ...
7,As per your request 'Melle Melle (Oru Minnamin...
8,WINNER!! As a valued network customer you have...
9,Had your mobile 11 months or more? U R entitle...


In [417]:
df_spam['v2'] = df_spam['v2'].astype('string')
# as a numerical value is present in the column so we have to convert the number to string to do the operations.

In [418]:
df_spam['lower_case'] = df_spam['v2'].str.lower()

In [419]:
df_spam['lower_case'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 5572 entries, 0 to 5571
Series name: lower_case
Non-Null Count  Dtype 
--------------  ----- 
5572 non-null   string
dtypes: string(1)
memory usage: 43.7 KB


## Punctuation Removal

In [420]:
df_spam['punctuation'] = df_spam['lower_case'].apply(remove_punctuation)

In [421]:
df_spam['punctuation'] = df_spam['punctuation'].apply(lambda x : x.replace('�', ''))

In [422]:
df_spam['punctuation'].head(10)

,punctuation
0,go until jurong point crazy available only in ...
1,ok lar joking wif u oni
2,free entry in 2 a wkly comp to win fa cup fina...
3,u dun say so early hor u c already then say
4,nah i dont think he goes to usf he lives aroun...
5,freemsg hey there darling its been 3 weeks now...
6,even my brother is not like to speak with me t...
7,as per your request melle melle oru minnaminun...
8,winner as a valued network customer you have b...
9,had your mobile 11 months or more u r entitled...


## Tokenization

In [423]:
df_spam['tokens'] = df_spam['punctuation'].apply(tokenization)

In [424]:
df_spam['tokens'].head(10)

,tokens
0,"[go, until, jurong, point, crazy, available, o..."
1,"[ok, lar, joking, wif, u, oni]"
2,"[free, entry, in, 2, a, wkly, comp, to, win, f..."
3,"[u, dun, say, so, early, hor, u, c, already, t..."
4,"[nah, i, dont, think, he, goes, to, usf, he, l..."
5,"[freemsg, hey, there, darling, its, been, 3, w..."
6,"[even, my, brother, is, not, like, to, speak, ..."
7,"[as, per, your, request, melle, melle, oru, mi..."
8,"[winner, as, a, valued, network, customer, you..."
9,"[had, your, mobile, 11, months, or, more, u, r..."


## Stop Words Removal

In [425]:
df_spam['stop_words'] = df_spam['tokens'].apply(remove_stopwords)

In [426]:
df_spam['stop_words'].head(10)

,stop_words
0,"[go, jurong, point, crazy, available, bugis, n..."
1,"[ok, lar, joking, wif, u, oni]"
2,"[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,"[u, dun, say, early, hor, u, c, already, say]"
4,"[nah, dont, think, goes, usf, lives, around, t..."
5,"[freemsg, hey, darling, 3, weeks, word, back, ..."
6,"[even, brother, like, speak, treat, like, aids..."
7,"[per, request, melle, melle, oru, minnaminungi..."
8,"[winner, valued, network, customer, selected, ..."
9,"[mobile, 11, months, u, r, entitled, update, l..."


## Stemming

In [427]:
df_spam['stem_words'] = df_spam['stop_words'].apply(stemming)

In [428]:
df_spam['stem_words'].head(10)

,stem_words
0,"[go, jurong, point, crazi, avail, bugi, n, gre..."
1,"[ok, lar, joke, wif, u, oni]"
2,"[free, entri, 2, wkli, comp, win, fa, cup, fin..."
3,"[u, dun, say, earli, hor, u, c, alreadi, say]"
4,"[nah, dont, think, goe, usf, live, around, tho..."
5,"[freemsg, hey, darl, 3, week, word, back, id, ..."
6,"[even, brother, like, speak, treat, like, aid,..."
7,"[per, request, mell, mell, oru, minnaminungint..."
8,"[winner, valu, network, custom, select, receiv..."
9,"[mobil, 11, month, u, r, entitl, updat, latest..."


## Lemmatization

In [429]:
df_spam['lemma_words'] = df_spam['stem_words'].apply(lemmatization)

In [430]:
df_spam['lemma_words'].head(10)

,lemma_words
0,"[go, jurong, point, crazi, avail, bugi, n, gre..."
1,"[ok, lar, joke, wif, u, oni]"
2,"[free, entri, 2, wkli, comp, win, fa, cup, fin..."
3,"[u, dun, say, earli, hor, u, c, alreadi, say]"
4,"[nah, dont, think, goe, usf, live, around, tho..."
5,"[freemsg, hey, darl, 3, week, word, back, id, ..."
6,"[even, brother, like, speak, treat, like, aid,..."
7,"[per, request, mell, mell, oru, minnaminungint..."
8,"[winner, valu, network, custom, select, receiv..."
9,"[mobil, 11, month, u, r, entitl, updat, latest..."


## Word Count

In [431]:
df_spam['lemma_word_count'] = df_spam['lemma_words'].apply(lambda x : len(x))
df_spam['initial_word_count'] = df_spam['punctuation'].apply(lambda x : len(x))

## DataFrame visualization

In [432]:
df_spam

,v1,v2,lower_case,punctuation,tokens,stop_words,stem_words,lemma_words,lemma_word_count,initial_word_count
0,ham,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ...",go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o...","[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre...","[go, jurong, point, crazi, avail, bugi, n, gre...",16,102
1,ham,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]","[ok, lar, joke, wif, u, oni]",6,23
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f...","[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...",23,149
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor... u c already then say...,u dun say so early hor u c already then say,"[u, dun, say, so, early, hor, u, c, already, t...","[u, dun, say, early, hor, u, c, already, say]","[u, dun, say, earli, hor, u, c, alreadi, say]","[u, dun, say, earli, hor, u, c, alreadi, say]",9,43
4,ham,"Nah I don't think he goes to usf, he lives aro...","nah i don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...,"[nah, i, dont, think, he, goes, to, usf, he, l...","[nah, dont, think, goes, usf, lives, around, t...","[nah, dont, think, goe, usf, live, around, tho...","[nah, dont, think, goe, usf, live, around, tho...",8,59
...,...,...,...,...,...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,this is the 2nd time we have tried 2 contact u...,this is the 2nd time we have tried 2 contact u...,"[this, is, the, 2nd, time, we, have, tried, 2,...","[2nd, time, tried, 2, contact, u, u, 750, poun...","[2nd, time, tri, 2, contact, u, u, 750, pound,...","[2nd, time, tri, 2, contact, u, u, 750, pound,...",20,151
5568,ham,Will �_ b going to esplanade fr home?,will �_ b going to esplanade fr home?,will b going to esplanade fr home,"[will, b, going, to, esplanade, fr, home]","[b, going, esplanade, fr, home]","[b, go, esplanad, fr, home]","[b, go, esplanad, fr, home]",5,34
5569,ham,"Pity, * was in mood for that. So...any other s...","pity, * was in mood for that. so...any other s...",pity was in mood for that soany other suggest...,"[pity, was, in, mood, for, that, soany, other,...","[pity, mood, soany, suggestions]","[piti, mood, soani, suggest]","[piti, mood, soani, suggest]",4,50
5570,ham,The guy did some bitching but I acted like i'd...,the guy did some bitching but i acted like i'd...,the guy did some bitching but i acted like id ...,"[the, guy, did, some, bitching, but, i, acted,...","[guy, bitching, acted, like, id, interested, b...","[guy, bitch, act, like, id, interest, buy, som...","[guy, bitch, act, like, id, interest, buy, som...",14,124


# Bag of Words

In [433]:
count_vectorizer_obj = CountVectorizer()

df_spam['clean_text'] = df_spam['lemma_words'].apply(lambda x : ' '.join(x))
df_spam.head(3)

,v1,v2,lower_case,punctuation,tokens,stop_words,stem_words,lemma_words,lemma_word_count,initial_word_count,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ...",go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o...","[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre...","[go, jurong, point, crazi, avail, bugi, n, gre...",16,102,go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]","[ok, lar, joke, wif, u, oni]",6,23,ok lar joke wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f...","[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...",23,149,free entri 2 wkli comp win fa cup final tkt 21...


In [434]:
count_vec = count_vectorizer_obj.fit_transform(df_spam['clean_text'])     # transforming the corpus to vector
count_vec.toarray()   # the vector representation to arrray

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [435]:
count_vec.shape   # return the dimension of the array

(5572, 7959)

# TF-IDF

- Term Frequency and Inverse Term Frequency
- Term Frequency measure how often the word appear in the document
- Inverse Document Frequency reduce the weight of common occuered word in multiple documents while increase the weight of rare words.

In [436]:
TF_IDF_vectorizer = TfidfVectorizer()   # object creation
tf_idf_vector = TF_IDF_vectorizer.fit_transform(df_spam['clean_text'])    # transform text to vectors
tf_idf_vector.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

# N-Grams Vectorization

In [437]:
df_spam['clean_tokens'] = df_spam['tokens'].apply(lambda x : ' '.join(x))
df_spam.head(3)

,v1,v2,lower_case,punctuation,tokens,stop_words,stem_words,lemma_words,lemma_word_count,initial_word_count,clean_text,clean_tokens
0,ham,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ...",go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o...","[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre...","[go, jurong, point, crazi, avail, bugi, n, gre...",16,102,go jurong point crazi avail bugi n great world...,go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]","[ok, lar, joke, wif, u, oni]",6,23,ok lar joke wif u oni,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f...","[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...",23,149,free entri 2 wkli comp win fa cup final tkt 21...,free entry in 2 a wkly comp to win fa cup fina...


In [438]:
bigram_vectorizer = CountVectorizer(ngram_range=(2,2))  # ngram parameter set to make sure we are getting bigrams only.
bigram_vectors = bigram_vectorizer.fit_transform(df_spam['clean_tokens'])
bigram_vectors.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [439]:
# Inference :
# Do not need to do all preprocessing steps (lemmatization, stop words removal, stemming) for n-grams. In fact, doing all of them usually ruins n-grams by breaking the natural flow and meaning of the phrases.

# Word2Vec

In [440]:
api.info()['models'].keys()   # word2vec model available in gensim library

dict_keys(['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis'])

In [441]:
# building Word2Vec neural network

model_w2v = Word2Vec(
    sentences=df_spam['tokens'],
    vector_size=100, # word embedding size (dimension of the vector rep)
    window=5,   # context information window (getting relationship)
    min_count=2, # Include all words appearing at least once ( ignore words with frequency < 2), to avoid the rare words.
    workers=4,    # cpu threads to spped up training
    sg=1,   # sg=0 means continuous bag of words and sg =1 means skipgrams
    epochs=30  # no of iterations ( how many times word2vec sees the data)
)

In [442]:
# function to match preserve the dimension of original dataframe.
def document_vector(tokens):
    valid_words = [word for word in tokens if word in model_w2v.wv]

    if len(valid_words) == 0:
        return np.zeros(model_w2v.vector_size)    # if word length is 0 in w2v vectors it will give 0's to maintain the dimension

    return np.mean(model_w2v.wv[valid_words], axis=0)

In [443]:
# building Word2Vec neural network - Continuous Bag of Words

model_w2v_cbow = Word2Vec(
    sentences=df_spam['tokens'],
    vector_size=100, # word embedding size (dimension of the vector rep)
    window=5,   # context information window (getting relationship)
    min_count=2, # Include all words appearing at least once ( ignore words with frequency < 2), to avoid the rare words.
    workers=4,    # cpu threads to spped up training
    sg=0,   # sg=0 means continuous bag of words and sg =1 means skipgrams
    epochs=30  # no of iterations ( how many times word2vec sees the data)
)

In [444]:
# function to match preserve the dimension of original dataframe.
def document_vector_cbow(tokens):
    valid_words = [word for word in tokens if word in model_w2v_cbow.wv]

    if len(valid_words) == 0:
        return np.zeros(model_w2v_cbow.vector_size)    # if word length is 0 in w2v vectors it will give 0's to maintain the dimension

    return np.mean(model_w2v_cbow.wv[valid_words], axis=0)

# Model Building and Evaluation
- Logistic Regression

In [445]:
df_spam['v1'].unique()

array(['ham', 'spam'], dtype=object)

## Target and Features

In [446]:
X = count_vec
y = df_spam['v1']

In [447]:
X1 = tf_idf_vector

In [448]:
X2 = bigram_vectors

In [449]:
X3 = np.array(df_spam["tokens"].apply(document_vector).tolist())
print(X3.shape)

(5572, 100)


In [450]:
X4 = np.array(df_spam["tokens"].apply(document_vector_cbow).tolist())
print(X4.shape)

(5572, 100)


## Train test split

In [451]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)

In [452]:
X_train1, X_test1, y_train, y_test = train_test_split(X1, y, random_state = 42, test_size = 0.2)

In [453]:
X_train2, X_test2, y_train, y_test = train_test_split(X2, y, random_state = 42, test_size = 0.2)

In [454]:
X_train3, X_test3, y_train, y_test = train_test_split(X3, y, random_state = 42, test_size = 0.2)

In [455]:
X_train4, X_test4, y_train, y_test = train_test_split(X4, y, random_state = 42, test_size = 0.2)

## Encoding the target

In [456]:
label_encoder_obj = LabelEncoder()
y_train = label_encoder_obj.fit_transform(y_train)
y_test = label_encoder_obj.transform(y_test)

## Log Model - Classification

In [457]:
# BagofWords

log_model = LogisticRegression()  # model object creation
log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)

In [458]:
# TF_IDF_Vectorization

log_model.fit(X_train1, y_train)
y_pred1 = log_model.predict(X_test1)

In [459]:
# N-Grams

log_model.fit(X_train2, y_train)
y_pred2 = log_model.predict(X_test2)

In [460]:
# SkipGrams (word2vec)

log_model.fit(X_train3, y_train)
y_pred3 = log_model.predict(X_test3)

In [461]:
# Continuous Bag of Words (word2vec)

log_model.fit(X_train4, y_train)
y_pred4 = log_model.predict(X_test4)

## Model Metrices

In [462]:
print(f'Accuracy of log model \t:  {(accuracy_score(y_test, y_pred)*100):.2f}%')
print(f'F1 Score of log model \t:  {(f1_score(y_test, y_pred)*100):.2f}%')
print(f'Recall of log model \t:  {(recall_score(y_test, y_pred)*100):.2f}%')
print(f"Precision of log model \t: {precision_score(y_test, y_pred) * 100:.2f}%")


Accuracy of log model 	:  97.85%
F1 Score of log model 	:  91.30%
Recall of log model 	:  84.00%
Precision of log model 	: 100.00%


In [463]:
# # Inferance :
# # - As the dataset is unbalanced where only 13%  are spam messages and the rest are non spam so we have to make the model better:
#       - one way is to make the non spam message count equal to the spam message count
#       - else we have to use the boosting model where it focus on the unbalanced dataset and make model more accurate.

In [464]:
print(f'Accuracy of log model using TF_IDF_vectors \t:  {(accuracy_score(y_test, y_pred1)*100):.2f}%')
print(f'F1 Score of log model using TF_IDF_vectors \t:  {(f1_score(y_test, y_pred1)*100):.2f}%')
print(f'Recall of log model using TF_IDF_vectors \t:  {(recall_score(y_test, y_pred1)*100):.2f}%')
print(f"Precision of log model using TF_IDF_vectors \t: {precision_score(y_test, y_pred1) * 100:.2f}%")

Accuracy of log model using TF_IDF_vectors 	:  94.44%
F1 Score of log model using TF_IDF_vectors 	:  74.80%
Recall of log model using TF_IDF_vectors 	:  61.33%
Precision of log model using TF_IDF_vectors 	: 95.83%


In [465]:
print(f'Accuracy of log model using Bigram vectors \t:  {(accuracy_score(y_test, y_pred2)*100):.2f}%')
print(f'F1 Score of log model using Bigram vectors \t:  {(f1_score(y_test, y_pred2)*100):.2f}%')
print(f'Recall of log model using Bigram vectors \t:  {(recall_score(y_test, y_pred2)*100):.2f}%')
print(f"Precision of log model using Bigram vectors \t: {precision_score(y_test, y_pred2) * 100:.2f}%")

Accuracy of log model using Bigram vectors 	:  95.34%
F1 Score of log model using Bigram vectors 	:  79.20%
Recall of log model using Bigram vectors 	:  66.00%
Precision of log model using Bigram vectors 	: 99.00%


In [466]:
print(f'Accuracy of log model using word2vec (skipgram)  :  {(accuracy_score(y_test, y_pred3)*100):.2f}%')
print(f'F1 Score of log model using word2vec (skipgram)  :  {(f1_score(y_test, y_pred3)*100):.2f}%')
print(f'Recall of log model using word2vec (skipgram)    :  {(recall_score(y_test, y_pred3)*100):.2f}%')
print(f"Precision of log model using word2vec (skipgram) : {precision_score(y_test, y_pred3) * 100:.2f}%")

Accuracy of log model using word2vec (skipgram)  :  97.76%
F1 Score of log model using word2vec (skipgram)  :  91.29%
Recall of log model using word2vec (skipgram)    :  87.33%
Precision of log model using word2vec (skipgram) : 95.62%


In [467]:
print(f'Accuracy of log model using word2vec (CBoW)  :  {(accuracy_score(y_test, y_pred4)*100):.2f}%')
print(f'F1 Score of log model using word2vec (CBoW)  :  {(f1_score(y_test, y_pred4, pos_label=1)*100):.2f}%')
print(f'Recall of log model using word2vec (CBoW)    :  {(recall_score(y_test, y_pred4, pos_label=1)*100):.2f}%')
print(f"Precision of log model using word2vec (CBoW) : {precision_score(y_test, y_pred4, pos_label=1) * 100:.2f}%")

Accuracy of log model using word2vec (CBoW)  :  96.68%
F1 Score of log model using word2vec (CBoW)  :  87.29%
Recall of log model using word2vec (CBoW)    :  84.67%
Precision of log model using word2vec (CBoW) : 90.07%


## Boosting Model

- As this is the unbalanced dataset we are going to use the boosting model

In [468]:
ada_boost_model = AdaBoostClassifier(
    estimator = LogisticRegression(),
    n_estimators = 20,
    learning_rate = 1
)

ada_boost_model.fit(X_train, y_train)
y_pred_boost = ada_boost_model.predict(X_test)


In [469]:


ada_boost_model.fit(X_train1, y_train)
y_pred_boost1 = ada_boost_model.predict(X_test1)


In [470]:


ada_boost_model.fit(X_train2, y_train)
y_pred_boost2 = ada_boost_model.predict(X_test2)


In [471]:


ada_boost_model.fit(X_train3, y_train)
y_pred_boost3 = ada_boost_model.predict(X_test3)


In [472]:


ada_boost_model.fit(X_train4, y_train)
y_pred_boost4 = ada_boost_model.predict(X_test4)


In [473]:
# Boosting Model Evaluation Metrices

print(f'Accuracy of Boosting model \t:  {(accuracy_score(y_test, y_pred_boost)*100):.2f}%')
print(f'F1 Score of Boosing model \t:  {(f1_score(y_test, y_pred_boost)*100):.2f}%')
print(f'Recall of Boosting model \t:  {(recall_score(y_test, y_pred_boost)*100):.2f}%')
print(f"Precision of Boosting model \t: {precision_score(y_test, y_pred_boost) * 100:.2f}%")

Accuracy of Boosting model 	:  97.94%
F1 Score of Boosing model 	:  92.26%
Recall of Boosting model 	:  91.33%
Precision of Boosting model 	: 93.20%


In [474]:
print(f'Accuracy of Boosting model using TF_IDF_vector \t:  {(accuracy_score(y_test, y_pred_boost1)*100):.2f}%')
print(f'F1 Score of Boosing model using TF_IDF_vector \t:  {(f1_score(y_test, y_pred_boost1)*100):.2f}%')
print(f'Recall of Boosting model using TF_IDF_vector \t:  {(recall_score(y_test, y_pred_boost1)*100):.2f}%')
print(f"Precision of Boosting model using TF_IDF_vector : {precision_score(y_test, y_pred_boost1) * 100:.2f}%")

Accuracy of Boosting model using TF_IDF_vector 	:  96.59%
F1 Score of Boosing model using TF_IDF_vector 	:  86.52%
Recall of Boosting model using TF_IDF_vector 	:  81.33%
Precision of Boosting model using TF_IDF_vector : 92.42%


In [475]:
print(f'Accuracy of Boosting model using bigram vectors :  {(accuracy_score(y_test, y_pred_boost2)*100):.2f}%')
print(f'F1 Score of Boosing model using bigram vectors \t:  {(f1_score(y_test, y_pred_boost2)*100):.2f}%')
print(f'Recall of Boosting model using bigram vectors \t:  {(recall_score(y_test, y_pred_boost2)*100):.2f}%')
print(f"Precision of Boosting model using bigram vectors : {precision_score(y_test, y_pred_boost2) * 100:.2f}%")

Accuracy of Boosting model using bigram vectors :  95.34%
F1 Score of Boosing model using bigram vectors 	:  79.20%
Recall of Boosting model using bigram vectors 	:  66.00%
Precision of Boosting model using bigram vectors : 99.00%


In [476]:
print(f'Accuracy of Boosting model using skipgram (word2vec) \t:  {(accuracy_score(y_test, y_pred_boost3)*100):.2f}%')
print(f'F1 Score of Boosing model using skipgram (word2vec) \t:  {(f1_score(y_test, y_pred_boost3)*100):.2f}%')
print(f'Recall of Boosting model using skipgram (word2vec) \t:  {(recall_score(y_test, y_pred_boost3)*100):.2f}%')
print(f"Precision of Boosting model using skipgram (word2vec) \t: {precision_score(y_test, y_pred_boost3) * 100:.2f}%")

Accuracy of Boosting model using skipgram (word2vec) 	:  96.95%
F1 Score of Boosing model using skipgram (word2vec) 	:  88.44%
Recall of Boosting model using skipgram (word2vec) 	:  86.67%
Precision of Boosting model using skipgram (word2vec) 	: 90.28%


In [477]:
print(f'Accuracy of Boosting model using CBOW (word2vec) \t:  {(accuracy_score(y_test, y_pred_boost4)*100):.2f}%')
print(f'F1 Score of Boosing model using CBOW (word2vec) \t:  {(f1_score(y_test, y_pred_boost4)*100):.2f}%')
print(f'Recall of Boosting model using CBOW (word2vec)  \t:  {(recall_score(y_test, y_pred_boost4)*100):.2f}%')
print(f"Precision of Boosting model using CBOW (word2vec) \t: {precision_score(y_test, y_pred_boost4) * 100:.2f}%")

Accuracy of Boosting model using CBOW (word2vec) 	:  96.77%
F1 Score of Boosing model using CBOW (word2vec) 	:  88.08%
Recall of Boosting model using CBOW (word2vec)  	:  88.67%
Precision of Boosting model using CBOW (word2vec) 	: 87.50%


In [478]:
# Inference:
# - Here we get better accuracy along with better recall (false negatives).